# Evaluación de Sistemas — Corpus Dialectal (modelos pequeños)
**Paper:** Dialectal Robustness in Spanish-to-English Speech Translation

Pipeline de validación con modelos ligeros — todos caben en T4 simultáneamente.

| Sistema | Modelo | VRAM aprox |
|---------|--------|------------|
| Whisper ASR + E2E | `openai/whisper-small` | ~500 MB |
| Cascade MT | `Helsinki-NLP/opus-mt-es-en` | ~300 MB |
| SeamlessM4T | `facebook/seamless-m4t-medium` | ~2.5 GB |
| **Total** | | **~3.3 GB** |

In [ ]:
!pip install -q transformers datasets sacrebleu soundfile librosa tqdm sentencepiece

In [ ]:
from google.colab import drive
import os, torch, gc
import numpy as np
import pandas as pd

drive.mount('/content/drive', force_remount=True)

BASE_DIR    = '/content/drive/MyDrive/MSR_Dialectal'
RESULTS_DIR = f'{BASE_DIR}/results'
AUDIO_DIR   = f'{BASE_DIR}/audio'
os.makedirs(RESULTS_DIR, exist_ok=True)

CORPUS_CSV = f'{BASE_DIR}/csv/corpus_dialectal_completo.csv'
device     = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Dispositivo: {device}')
print(f'VRAM total: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## 1. Cargar corpus y audio

In [ ]:
import urllib.request, zipfile, glob
from datasets import load_dataset, concatenate_datasets

corpus = pd.read_csv(CORPUS_CSV)
print(f'Corpus: {len(corpus)} muestras')
print(corpus.groupby('dialect')['transcription'].count())

# ── Audio OpenSLR (pe, pr, ve) ──────────────────────────────────────────────
OPENSLR_AUDIO = {
    'pe': [
        ('https://www.openslr.org/resources/73/es_pe_female.zip', 'es_pe_female'),
        ('https://www.openslr.org/resources/73/es_pe_male.zip',   'es_pe_male'),
    ],
    'pr': [('https://www.openslr.org/resources/74/es_pr_female.zip', 'es_pr_female')],
    've': [
        ('https://www.openslr.org/resources/75/es_ve_female.zip', 'es_ve_female'),
        ('https://www.openslr.org/resources/75/es_ve_male.zip',   'es_ve_male'),
    ],
}

openslr_index = {}
for dialect, files in OPENSLR_AUDIO.items():
    out_dir = f'{AUDIO_DIR}/{dialect}'
    os.makedirs(out_dir, exist_ok=True)
    for url, name in files:
        zip_path = f'{AUDIO_DIR}/{name}.zip'
        wavs_existing = glob.glob(f'{out_dir}/**/*.wav', recursive=True)
        if not wavs_existing and not os.path.exists(zip_path):
            print(f'Descargando {name}...')
            urllib.request.urlretrieve(url, zip_path)
            with zipfile.ZipFile(zip_path, 'r') as z:
                z.extractall(out_dir)
            os.remove(zip_path)
    for w in glob.glob(f'{out_dir}/**/*.wav', recursive=True):
        openslr_index[os.path.splitext(os.path.basename(w))[0]] = w
    print(f'  {dialect}: {len([k for k in openslr_index if k[:2]==dialect])} WAVs')

# ── Audio HuggingFace (co, cl, ar) ──────────────────────────────────────────
HF_DATASETS = {
    'co': 'ylacombe/google-colombian-spanish',
    'cl': 'ylacombe/google-chilean-spanish',
    'ar': 'ylacombe/google-argentinian-spanish',
}
hf_audio = {}
for dialect, hf_id in HF_DATASETS.items():
    parts = []
    for config in ['female', 'male']:
        try:
            parts.append(load_dataset(hf_id, config, split='train'))
        except Exception:
            pass
    if parts:
        ds = concatenate_datasets(parts)
        text_col = next((c for c in ['text','sentence','transcription'] if c in ds.column_names), ds.column_names[1])
        hf_audio[dialect] = {row[text_col]: row['audio'] for row in ds}
        print(f'  {dialect}: {len(hf_audio[dialect])} audios HF')

print('Audio listo.')

In [ ]:
import soundfile as sf
import librosa

def get_audio(row, target_sr=16000):
    dialect = row['dialect']
    if dialect in hf_audio:
        obj = hf_audio[dialect].get(row['transcription'])
        if obj is None:
            return None
        arr = np.array(obj['array'], dtype=np.float32)
        sr  = obj['sampling_rate']
    else:
        path = openslr_index.get(row['file_id'])
        if path is None:
            return None
        arr, sr = sf.read(path, dtype='float32')
        if arr.ndim > 1:
            arr = arr.mean(axis=1)
    if sr != target_sr:
        arr = librosa.resample(arr, orig_sr=sr, target_sr=target_sr)
    return arr

# Test cobertura
found = sum(1 for _, row in corpus.iterrows() if get_audio(row) is not None)
print(f'Audio disponible: {found}/{len(corpus)} muestras')

## 2. Cargar todos los modelos

In [ ]:
from transformers import (
    WhisperProcessor, WhisperForConditionalGeneration,
    MarianMTModel, MarianTokenizer,
    AutoProcessor, SeamlessM4TModel,
)

# Whisper small — sirve para ASR y E2E (mismo modelo, distinto task)
print('Cargando Whisper small...')
whisper_processor = WhisperProcessor.from_pretrained('openai/whisper-small')
whisper_model     = WhisperForConditionalGeneration.from_pretrained(
    'openai/whisper-small', torch_dtype=torch.float16
).to(device)
whisper_model.eval()
print(f'  OK | VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB')

# Helsinki-NLP opus-mt-es-en — cascada MT
print('Cargando Helsinki-NLP opus-mt-es-en...')
mt_tokenizer = MarianTokenizer.from_pretrained('Helsinki-NLP/opus-mt-es-en')
mt_model     = MarianMTModel.from_pretrained(
    'Helsinki-NLP/opus-mt-es-en', torch_dtype=torch.float16
).to(device)
mt_model.eval()
print(f'  OK | VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB')

# SeamlessM4T medium — E2E moderno
print('Cargando SeamlessM4T medium...')
seamless_processor = AutoProcessor.from_pretrained('facebook/seamless-m4t-medium')
seamless_model     = SeamlessM4TModel.from_pretrained(
    'facebook/seamless-m4t-medium', torch_dtype=torch.float16
).to(device)
seamless_model.eval()
print(f'  OK | VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB')

print('\nTodos los modelos cargados.')

## 3. Funciones de inferencia

In [ ]:
def run_asr(audio):
    inputs = whisper_processor(audio, sampling_rate=16000, return_tensors='pt').to(device)
    with torch.no_grad():
        ids = whisper_model.generate(**inputs, language='es', task='transcribe')
    return whisper_processor.batch_decode(ids, skip_special_tokens=True)[0]

def run_e2e(audio):
    inputs = whisper_processor(audio, sampling_rate=16000, return_tensors='pt').to(device)
    with torch.no_grad():
        ids = whisper_model.generate(**inputs, language='en', task='translate')
    return whisper_processor.batch_decode(ids, skip_special_tokens=True)[0]

def run_cascade_mt(text):
    inputs = mt_tokenizer([text], return_tensors='pt', truncation=True, max_length=512).to(device)
    with torch.no_grad():
        ids = mt_model.generate(**inputs)
    return mt_tokenizer.decode(ids[0], skip_special_tokens=True)

def run_seamless(audio):
    inputs = seamless_processor(audios=audio, sampling_rate=16000, return_tensors='pt').to(device)
    with torch.no_grad():
        ids = seamless_model.generate(**inputs, tgt_lang='eng')
    return seamless_processor.decode(ids[0].tolist(), skip_special_tokens=True)

print('Funciones listas.')

## 4. Loop de evaluación

In [ ]:
from tqdm import tqdm
import json

CHECKPOINT = f'{RESULTS_DIR}/checkpoint.json'

records = []
if os.path.exists(CHECKPOINT):
    with open(CHECKPOINT) as f:
        records = json.load(f)
    done_ids = {r['file_id'] for r in records}
    print(f'Retomando: {len(records)} ya procesadas')
else:
    done_ids = set()

pending = corpus[~corpus['file_id'].isin(done_ids)]
print(f'Pendientes: {len(pending)}')

for _, row in tqdm(pending.iterrows(), total=len(pending)):
    audio = get_audio(row)
    if audio is None:
        continue

    rec = {
        'file_id':   row['file_id'],
        'dialect':   row['dialect'],
        'ref':       row['reference_en'],
        'asr':       '',
        'e2e':       '',
        'cascade':   '',
        'seamless':  '',
    }

    try:
        rec['asr'] = run_asr(audio)
    except Exception as e:
        pass

    try:
        rec['e2e'] = run_e2e(audio)
    except Exception as e:
        pass

    try:
        if rec['asr']:
            rec['cascade'] = run_cascade_mt(rec['asr'])
    except Exception as e:
        pass

    try:
        rec['seamless'] = run_seamless(audio)
    except Exception as e:
        pass

    records.append(rec)

    # Checkpoint cada 100 muestras
    if len(records) % 100 == 0:
        with open(CHECKPOINT, 'w') as f:
            json.dump(records, f, ensure_ascii=False)

# Guardar final
with open(CHECKPOINT, 'w') as f:
    json.dump(records, f, ensure_ascii=False)

df_results = pd.DataFrame(records)
df_results.to_csv(f'{RESULTS_DIR}/all_results.csv', index=False)
print(f'\nGuardado: {len(df_results)} muestras')

## 5. Métricas finales

In [ ]:
import sacrebleu

DIALECT_NAMES = {'pe':'Peru','pr':'Puerto Rico','ve':'Venezuela',
                 'co':'Colombia','cl':'Chile','ar':'Argentina'}
SYSTEMS = {
    'e2e':      'Whisper E2E',
    'cascade':  'Cascade (Whisper + Helsinki)',
    'seamless': 'SeamlessM4T medium',
}

rows = []
for sys_col, sys_name in SYSTEMS.items():
    for dialect, grp in df_results.groupby('dialect'):
        pairs = [(h, r) for h, r in zip(grp[sys_col], grp['ref']) if h]
        if not pairs:
            continue
        hyps, refs = zip(*pairs)
        bleu = sacrebleu.corpus_bleu(list(hyps), [list(refs)]).score
        chrf = sacrebleu.corpus_chrf(list(hyps), [list(refs)]).score
        rows.append({
            'Sistema':  sys_name,
            'Dialecto': DIALECT_NAMES.get(dialect, dialect),
            'N':        len(pairs),
            'BLEU':     round(bleu, 2),
            'chrF':     round(chrf, 2),
        })

df_metrics = pd.DataFrame(rows)

pivot = df_metrics.pivot(index='Sistema', columns='Dialecto', values='BLEU')
pivot['Promedio'] = pivot.mean(axis=1).round(2)
pivot = pivot.sort_values('Promedio', ascending=False)

print('=== BLEU corpus-level por Sistema × Dialecto ===')
print(pivot.to_string())

df_metrics.to_csv(f'{RESULTS_DIR}/metrics_small_models.csv', index=False)
pivot.to_csv(f'{RESULTS_DIR}/pivot_bleu_small.csv')
print(f'\nGuardado en {RESULTS_DIR}/')